# Dentate RL bootcamp on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xpitfire/dentate-bootcamp/blob/main/dentate_bootcamp.ipynb)

**Verify before you reward.** Dentate fine-tunes a small model on a verifiable task family, measures whether it can
already produce checkable answers, and runs reinforcement learning (GRPO) only when that measurement says a reward
exists. Every run ends with a held-out score and the verdict behind it.

This notebook is the hands-on version of the bootcamp:

| Section | What you do |
|---|---|
| 1. Install | `pip install "dentate[demo]"` (also installs the `cortex` CLI) |
| 2. Setup | materialise the bundled starter project and the pinned tokenizer, offline |
| 3. Configure | pick model size, task family, budgets and RL knobs with the form fields |
| 4. Train | run SFT → support gate → GRPO → frozen evaluation with live progress |
| 5. Inspect | plot the training curves, the support gate and the before/after evaluation |
| 6. Sweep | optional: repeat the run over a few seeds or budgets and compare |
| 7. Dashboard | open the Dentate console (lab, results, terminal) through Colab's port proxy |

Everything after the install is offline: no model weights are downloaded and no teacher model is called. The demo
ships **Spiral**, a small looped transformer, as its reference architecture; any architecture registered with Dentate
runs through the same pipeline.

> A **failed support gate is a valid result**. A cold ~0.1M-parameter model often cannot sample a single verifiable
> answer after a short SFT; the pipeline then refuses to run RL and says why, instead of reporting a number nobody
> can trust. Section 6 shows what to change to move the gate.

Runtime: a CPU runtime is enough (`Runtime → Change runtime type → CPU`). The default configuration below takes
2–6 minutes.

## 1. Install

`dentate[demo]` bundles the pipeline, the Spiral reference architecture (`spiral-lm`), a CPU build of PyTorch, the
SmolLM tokenizer and a starter project. The `cortex` CLI is installed alongside (Python ≥ 3.12) so the terminal in
the dashboard has it on its PATH.

In [ ]:
# 1. Install. Colab ships torch already; elsewhere the `demo` extra pulls in a CPU torch.
import importlib.util, os, subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

if importlib.util.find_spec("dentate") is None:
    pip("dentate[demo]")
DENTATE = [sys.executable, "-m", "dentate.cli"]      # the `dentate` console script, pinned to THIS kernel's interpreter
BIN = os.path.dirname(sys.executable)

def version(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=60).stdout.strip() or "(no output)"
    except Exception as exc:  # noqa: BLE001 — a missing optional tool is reported, not fatal
        return f"unavailable ({type(exc).__name__})"

import importlib.metadata as md_
print("dentate ", md_.version("dentate"))
print("spiral-lm", md_.version("spiral-lm"))
print("torch   ", md_.version("torch"))
print("cortex  ", version([os.path.join(BIN, "cortex"), "--version"]))

## 2. Set up the demo bundle

`dentate demo init` copies the pinned tokenizer and the starter project to `~/.dentate/demo` (or `$DENTATE_HOME/demo`)
and verifies their SHA-256 against the bundled manifest. The `export` line it prints is propagated into this kernel so
every later cell finds the tokenizer without downloading anything.

In [ ]:
# 2. Materialize the demo bundle (tokenizer + starter project) and pin DENTATE_TOKENIZER_DIR for this kernel.
import os, pathlib, subprocess

subprocess.run([*DENTATE, "demo", "doctor"], check=True)
init = subprocess.run([*DENTATE, "demo", "init"], check=True, capture_output=True, text=True).stdout
print(init)
for line in init.splitlines():
    if line.startswith("export "):
        key, _, value = line[len("export "):].partition("=")
        os.environ[key] = value           # the CLI sets it for ITS process; propagate the export line into this kernel
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
HOME = pathlib.Path(os.environ.get("DENTATE_HOME") or pathlib.Path.home() / ".dentate")   # same root the CLI uses
EXPERIMENTS = HOME / "experiments"; EXPERIMENTS.mkdir(parents=True, exist_ok=True)
print("experiments will be written under", EXPERIMENTS)

## 3. Configure the experiment

Every field below is a form in Colab (double-click the cell to see the code). The values are validated against the
same schema the lab UI and the CLI use, so anything you can set here you can also set in a `.dentate` project file.

**Model** — `width` is the hidden size of Spiral's single shared block, `embedding` the factorised input embedding,
`loops` how many times that block is iterated (depth from iteration, not from stacking layers). Wider/deeper models
learn more per step but train slower on CPU.

**Task family** — `arith` (small arithmetic), `modular` (modular arithmetic), `count`, `parity`, `compare`. Each task has
a verifier: the reward is *checked*, never guessed. Small task spaces can run out of unseen content; the pipeline then
reports `insufficient_heldout` instead of scoring on training data.

**Budgets** — `sft_steps` supervised steps on verified demonstrations; `grpo_steps` RL steps that only run if the
support gate passes; `batch_size` per step; `samples` completions per prompt during GRPO and the gate (pass@k).

**RL knobs** — `temperature` for sampling, `kl_beta` the leash keeping the policy near the SFT model,
`rl_learning_rate` the GRPO step size. `eval_tasks` sets the size of the frozen, content-disjoint held-out set that
produces the before/after numbers; `max_new_tokens` caps generation length.

In [ ]:
# 3. Experiment configuration (Colab renders these as a form; elsewhere they are plain assignments).
#@title Experiment configuration { run: "auto", display-mode: "form" }
project_name = "My Spiral experiment"  #@param {type:"string"}

#@markdown ### Model
width = 64        #@param [32, 64, 128] {type:"raw"}
embedding = 32    #@param [16, 32, 64] {type:"raw"}
loops = 2         #@param {type:"slider", min:1, max:8, step:1}

#@markdown ### Task family and seed
kind = "arith"    #@param ["arith", "modular", "count", "parity", "compare"]
seed = 42         #@param {type:"integer"}

#@markdown ### Budgets
sft_steps = 100   #@param {type:"slider", min:1, max:500, step:1}
grpo_steps = 20   #@param {type:"slider", min:1, max:100, step:1}
batch_size = 2    #@param {type:"slider", min:1, max:4, step:1}
samples = 4       #@param [2, 4, 8] {type:"raw"}
eval_tasks = 8    #@param {type:"slider", min:2, max:16, step:1}

#@markdown ### Optimisation and sampling
learning_rate = 0.001       #@param {type:"number"}
rl_learning_rate = 0.00001  #@param {type:"number"}
temperature = 0.8           #@param {type:"slider", min:0.1, max:1.5, step:0.1}
kl_beta = 0.04              #@param {type:"slider", min:0.0, max:0.2, step:0.01}
max_new_tokens = 64         #@param {type:"slider", min:16, max:128, step:8}

import json, re
from dentate.bootcamp.project import Project

project = Project.model_validate({
    "name": project_name,
    "architecture": {"width": width, "embedding": embedding, "loops": loops},
    "training": {
        "kind": kind, "seed": seed, "sft_steps": sft_steps, "grpo_steps": grpo_steps, "batch_size": batch_size,
        "samples": samples, "eval_tasks": eval_tasks, "learning_rate": learning_rate,
        "rl_learning_rate": rl_learning_rate, "temperature": temperature, "kl_beta": kl_beta,
        "max_new_tokens": max_new_tokens,
    },
})
slug = re.sub(r"[^a-z0-9]+", "-", project_name.lower()).strip("-") or "experiment"
PROJECT_FILE = HOME / "demo" / f"{slug}.dentate"
PROJECT_FILE.write_text(project.model_dump_json(indent=2))
print("validated project written to", PROJECT_FILE)
print(json.dumps(project.model_dump(), indent=2))

## 4. Train

The run below is exactly `dentate demo run --project <file>`. It prints one JSON line per progress event; the cell
turns those into a compact log and keeps the per-step metrics for the plots. Stages:

1. **sft** — supervised fine-tuning on verified demonstrations.
2. **support** — sample `samples` completions for each held-out prompt; measure pass@1, pass@k and the *gap* between
   them. A gap means RL has something to harvest.
3. **grpo** — group-relative policy optimisation, only if the gate says a reward exists.
4. **frozen_evaluation** — accuracy on the content-disjoint held-out set, before and after RL.

In [ ]:
# 4. Train the configured project with live progress. The last line names the result package.
import json, subprocess, time

OUT = EXPERIMENTS / time.strftime(f"colab-{slug}-%Y%m%d-%H%M%S")
proc = subprocess.Popen([*DENTATE, "demo", "run", "--project", str(PROJECT_FILE), "--out", str(OUT)],
                        stdout=subprocess.PIPE, text=True, bufsize=1)
events, stage, t0 = [], None, time.time()
for line in proc.stdout:
    try:
        state = json.loads(line)
    except ValueError:
        print(line.rstrip()); continue
    events.append(state)
    if "message" in state:                                   # trainer log lines (loss / accuracy per iteration)
        print(f"[{state.get('elapsed_seconds', 0):>7.1f}s] {state['message']}")
    elif "out" in state:                                     # the final summary line
        print(f"finished: {state['status']} → {state['out']}")
    elif state.get("stage") != stage:                        # stage transitions: sft → support → grpo → frozen_evaluation
        stage = state.get("stage")
        print(f"[{state.get('elapsed_seconds', 0):>7.1f}s] stage: {stage}")
assert proc.wait() == 0, "demo run failed — see the log above"
results = json.loads((OUT / "results.json").read_text())
print()
print("status             :", results["status"])
print("gate verdict       :", (results.get("gate") or {}).get("verdict"))
before, after = results.get("frozen_before") or {}, results.get("frozen_after") or {}
print("frozen eval before :", before.get("verify_acc"))
print("frozen eval after  :", after.get("verify_acc", "absent (gate failed or GRPO did not run)"))
print("result package     :", OUT / "result.dentate")

## 5. Inspect the run

Three views of the same run:

- **Training curves** — loss and token accuracy per SFT step (and GRPO reward if it ran). Watch for the loss
  flattening: a run that is still dropping steeply when SFT ends simply needed more `sft_steps`.
- **Support gate** — pass@1 vs pass@k per task depth. RL can only reward what the model already samples sometimes;
  pass@k > pass@1 is the *harvestable gap*. `rlvr_ready` is the gate's decision.
- **Frozen evaluation** — verified accuracy on unseen tasks before and after GRPO. Only this number is a claim about
  generalisation; everything else is training-set signal.

In [ ]:
# 5a. Training curves from the progress events captured above.
train = [e["latest"] for e in events if "latest" in e and e["latest"].get("channel") == "train"]
metrics = [e["latest"] for e in events if "latest" in e and e["latest"].get("channel") == "metrics"]
try:
    import matplotlib.pyplot as plt
except ImportError:                       # plain-Linux headless runs without matplotlib still complete
    plt = None
    print("matplotlib not installed; skipping plots (pip install matplotlib)")

if plt and train:
    rows = [t for t in train if t.get("split") == "train"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    axes[0].plot([r["step"] for r in rows], [r["loss"] for r in rows], color="#2a9d8f")
    axes[0].set_title("SFT loss"); axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy")
    axes[1].plot([r["step"] for r in rows], [r.get("acc", 0.0) for r in rows], color="#e76f51", label="token acc (train)")
    evals = [t for t in train if t.get("split") == "eval"]
    if evals:
        axes[1].plot([r["step"] for r in evals], [r.get("verify_acc", 0.0) for r in evals], "o-", color="#264653", label="verified acc (eval)")
    axes[1].set_ylim(0, 1); axes[1].set_title("accuracy"); axes[1].set_xlabel("step"); axes[1].legend(loc="upper left")
    for ax in axes: ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
elif not train:
    print("no per-step training events were captured")

In [ ]:
# 5b. The support gate and the frozen evaluation.
support = results.get("support") or {}
rows = support.get("rows") or []
summary = support.get("summary") or results.get("gate") or {}
print("gate:", "RLVR ready" if summary.get("rlvr_ready") else "RLVR not ready", "·", summary.get("verdict"))
for r in rows:
    print(f"  {r['kind']} depth={r['depth']}: pass@1={r['pass@1']:.2f} pass@{r['n_samples']}={r.get('pass@%d' % r['n_samples'], r.get('pass@k', 0.0)):.2f} "
          f"gap={r['gap']:.2f} coverage={r['coverage']:.2f} ({r['n_tasks']} tasks)")

if plt:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    labels = [f"{r['kind']} d{r['depth']}" for r in rows] or ["(no rows)"]
    p1 = [r["pass@1"] for r in rows] or [0]
    pk = [r.get("pass@%d" % r["n_samples"], r.get("pass@k", 0.0)) for r in rows] or [0]
    x = range(len(labels))
    axes[0].bar([i - 0.2 for i in x], p1, width=0.4, label="pass@1", color="#264653")
    axes[0].bar([i + 0.2 for i in x], pk, width=0.4, label="pass@k", color="#2a9d8f")
    axes[0].set_xticks(list(x)); axes[0].set_xticklabels(labels); axes[0].set_ylim(0, 1)
    axes[0].set_title("support gate (harvestable gap = pass@k − pass@1)"); axes[0].legend()
    b = (results.get("frozen_before") or {}).get("verify_acc")
    a = (results.get("frozen_after") or {}).get("verify_acc")
    axes[1].bar(["before GRPO", "after GRPO"], [b or 0.0, a or 0.0], color=["#8d99ae", "#e76f51"])
    axes[1].set_ylim(0, 1); axes[1].set_title("frozen held-out verified accuracy")
    if a is None:
        axes[1].text(1, 0.05, "absent\n(GRPO did not run)", ha="center", color="#555")
    for ax in axes: ax.grid(axis="y", alpha=0.25)
    plt.tight_layout(); plt.show()

In [ ]:
# 5c. Everything the run recorded: metrics table, provenance, and the files in the result package.
import json, zipfile
try:
    import pandas as pd
except ImportError:
    pd = None
metrics_rows = json.loads((OUT / "metrics.json").read_text()).get("rows", [])
if pd is not None and metrics_rows:
    display(pd.DataFrame(metrics_rows).tail(8))
else:
    for r in metrics_rows[-5:]: print(r)
prov = json.loads((OUT / "provenance.json").read_text())
print("provenance keys:", sorted(prov)[:12])
with zipfile.ZipFile(OUT / "result.dentate") as z:
    names = z.namelist()
print(f"result.dentate: {len(names)} members, e.g. {names[:6]}")
print("This package is what the lab imports and what a published result exposes for download.")

## 6. Move the gate: a small sweep (optional)

The default starter usually ends in `gate_failed`: after 100 SFT steps a cold model rarely samples a verifiable answer.
Two levers move it: **more SFT** (the model needs support before RL can reward anything) and a **smaller task space**
(`kind`, `eval_tasks`). The sweep below re-runs the configured project over a list of `sft_steps` (or seeds) and plots
the gate outcome and the frozen evaluation for each. Each run costs roughly the time of section 4.

In [ ]:
# 6. Optional sweep over one knob. Set run_sweep to True in the form and pick the axis.
#@title Sweep { run: "auto", display-mode: "form" }
run_sweep = False            #@param {type:"boolean"}
sweep_axis = "sft_steps"     #@param ["sft_steps", "seed", "loops", "kl_beta"]
sweep_values = "50, 150, 300"  #@param {type:"string"}

import copy, json, subprocess, time
sweep = []
if run_sweep:
    values = [v.strip() for v in sweep_values.split(",") if v.strip()]
    for raw in values:
        variant = copy.deepcopy(project.model_dump())
        value = float(raw) if "." in raw else int(raw)
        (variant["training"] if sweep_axis != "loops" else variant["architecture"])[sweep_axis] = value
        variant["name"] = f"{project.name} · {sweep_axis}={raw}"
        vfile = HOME / "demo" / f"{slug}-{sweep_axis}-{raw}.dentate"
        vfile.write_text(Project.model_validate(variant).model_dump_json())
        vout = EXPERIMENTS / time.strftime(f"sweep-{slug}-{sweep_axis}-{raw}-%H%M%S")
        t0 = time.time()
        subprocess.run([*DENTATE, "demo", "run", "--project", str(vfile), "--out", str(vout), "--quiet"], check=True)
        res = json.loads((vout / "results.json").read_text())
        sweep.append({sweep_axis: raw, "status": res["status"], "rlvr_ready": (res.get("gate") or {}).get("rlvr_ready"),
                      "before": (res.get("frozen_before") or {}).get("verify_acc"),
                      "after": (res.get("frozen_after") or {}).get("verify_acc"), "seconds": round(time.time() - t0)})
        print(sweep[-1])
    if plt and sweep:
        fig, ax = plt.subplots(figsize=(6, 3.2))
        xs = [str(s[sweep_axis]) for s in sweep]
        ax.plot(xs, [s["before"] or 0 for s in sweep], "o-", label="frozen before", color="#8d99ae")
        ax.plot(xs, [s["after"] if s["after"] is not None else float("nan") for s in sweep], "o-", label="frozen after", color="#e76f51")
        for i, s in enumerate(sweep):
            ax.annotate(s["status"], (i, (s["before"] or 0) + 0.03), ha="center", fontsize=8)
        ax.set_ylim(0, 1); ax.set_xlabel(sweep_axis); ax.legend(); ax.grid(alpha=0.25)
        plt.tight_layout(); plt.show()
else:
    print("sweep skipped (set run_sweep = True to run it)")

## 7. Open the Dentate console

`dentate serve` starts the same site you get with `pip install "dentate[demo]"` on your laptop: the **lab** (edit and
launch projects, import `result.dentate` packages, download results), **results**, and a **terminal** dock bound to
this runtime (with `dentate` and `cortex` on its PATH). On Colab the browser cannot reach the VM's loopback address, so
the server admits Colab's port-proxy hostname and the link below carries the one-time launch token that unlocks the
terminal. The site refuses to be embedded in a frame, so it opens in its own tab.

In [ ]:
# 7. Start the Dentate site and open it. Outside Colab this starts the server on loopback and prints the local URL.
import socket, subprocess, sys, time
from urllib.parse import urlsplit

PORT = 8793
def listening(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

proxy_url = None
try:
    from google.colab.output import eval_js                      # Colab only
    proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
except ImportError:
    pass

server = None
if not listening(PORT):
    cmd = [*DENTATE, "serve", "--port", str(PORT)]
    if proxy_url:
        cmd += ["--proxy-host", urlsplit(proxy_url).hostname]
    server = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    for _ in range(60):
        if listening(PORT):
            break
        time.sleep(0.5)
assert listening(PORT), "dentate serve did not come up on port %d" % PORT

# The terminal dock unlocks with the launch secret the server wrote 0600 to <DENTATE_HOME>/.bootcamp/terminal-token.
token_file = HOME / ".bootcamp" / "terminal-token"
for _ in range(20):
    if token_file.is_file():
        break
    time.sleep(0.25)
launch = f"?token={token_file.read_text().strip()}" if token_file.is_file() else ""

if proxy_url:
    from IPython.display import HTML, display
    display(HTML(f'<p style="font-size:1.1em"><b>Open Dentate in a new tab:</b> <a href="{proxy_url}{launch}" target="_blank">{proxy_url}</a></p>'
                 '<p>Lab → import <code>result.dentate</code> from section 4, or launch a new run. The terminal dock at the bottom '
                 'is a real shell on this runtime.</p>'))
else:
    print(f"Not on Colab: open http://127.0.0.1:{PORT}/{launch} in your browser.")
print("result package to import in the lab:", OUT / "result.dentate")

### What to try in the console

- **Lab → Load .dentate project**: open the file written in section 3, tweak a field, start a run from the UI and
  watch the same stages stream in.
- **Terminal dock** (bottom): `dentate demo doctor`, `cortex --version`, or `python -m dentate.bootcamp run --project
  <file> --out <dir>` for a run outside the notebook. `cortex code` launches a governed coding session with the
  `omp` harness by default.
- **Results → Public**: on the hosted site (https://dentate.cortex.a2olabs.com) a finished experiment can be published
  to the public results page; locally it stays in your data directory.

Student guide: [BOOTCAMP.md](https://github.com/Xpitfire/dentate-bootcamp/blob/main/BOOTCAMP.md) · package:
[pypi.org/project/dentate](https://pypi.org/project/dentate/) · CLI bundles:
[releases](https://github.com/Xpitfire/dentate-bootcamp/releases/latest).

In [ ]:
# 8. (optional) Stop the background server when you are done.
if 'server' in globals() and server is not None and server.poll() is None:
    server.terminate()
    print("server stopped")